# R2: Brainstorm p(True)

**Goal:** Implement brainstorm p(True) (the model sees its own sampled answers before judging)
and the shared sanity test function `run_sanity()` used by R1, R2 and R3.

Model so far: `Qwen/Qwen2.5-0.5B-Instruct`. Run from `experiments/llm-probly` (the setup cell does this automatically).

## Setup

In [1]:
import os, sys, json
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from probly.representer.sampler.huggingface import HFTextGenerationSampler, load_model

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
torch.manual_seed(42)

print("working dir:", os.getcwd())

working dir: /Users/elcinekinsimsir/projects/probly-fork/experiments/llm-probly


## Implementation

### Step 1: Are "True" and " True" the same token?

In [2]:
from transformers import AutoTokenizer

for name in ["HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(name)
    print("  'True'  ->", tok("True", add_special_tokens=False)["input_ids"])
    print("  ' True' ->", tok(" True", add_special_tokens=False)["input_ids"])

HuggingFaceTB/SmolLM2-135M-Instruct
  'True'  -> [2787]
  ' True' -> [3635]
Qwen/Qwen2.5-0.5B-Instruct
  'True'  -> [2514]
  ' True' -> [3007]


**Finding: "True" vs " True" tokens.**
In both SmolLM2-135M and Qwen2.5-0.5B, "True" and " True" are different single tokens
(SmolLM2: 2787 vs 3635, Qwen: 2514 vs 3007). LM-Polygraph's `BasePromptCalculator`
tokenizes "True" without a leading space, but after "The possible answer is:" the model
most likely generates " True". So LM-Polygraph may be reading the score of a token the
model would not produce. To confirm: check the top next-token predictions (shared with R1).

### Step 2: What does the model actually predict after the prompt?

In [3]:
from probly.representer.sampler.huggingface import load_model

model, tok = load_model("Qwen/Qwen2.5-0.5B-Instruct")
model.eval()

prompt = (
    "Question: What is the capital of France?\n"
    "Here are some ideas that were brainstormed: Paris, Paris, Lyon, Paris, Marseille\n"
    "Possible answer: Paris\n"
    "Is the possible answer:\n"
    " (A) True\n"
    " (B) False\n"
    "The possible answer is:"
)

def top_next_tokens(text, k=10):
    ids = tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**ids).logits[0, -1]
    probs = torch.softmax(logits, dim=0)
    top = torch.topk(probs, k)
    for p, i in zip(top.values, top.indices):
        print(f"  {repr(tok.decode([i])):>12}  id={i.item():>6}  p={p.item():.3f}")

print("Without chat template:")
top_next_tokens(prompt)

print("\nWith chat template:")
chat_text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                    tokenize=False, add_generation_prompt=True)
top_next_tokens(chat_text)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Without chat template:
          ' ('  id=   320  p=0.699
          ' A'  id=   362  p=0.291
         ' \n'  id=   715  p=0.006
          ' B'  id=   425  p=0.003
       ' True'  id=  3007  p=0.001
        '  \n'  id=  2303  p=0.000
       ' \n\n'  id=  4710  p=0.000
           ' '  id=   220  p=0.000
        ' Yes'  id=  7414  p=0.000
  '<|endoftext|>'  id=151643  p=0.000

With chat template:
         'The'  id=   785  p=0.475
          '(A'  id=  4346  p=0.369
          '(B'  id=  5349  p=0.057
           'A'  id=    32  p=0.024
           'B'  id=    33  p=0.011
           '('  id=     7  p=0.009
         'Yes'  id=  9454  p=0.008
          '(C'  id=  3025  p=0.007
          'Is'  id=  3872  p=0.005
      'Answer'  id= 16141  p=0.004


**Finding: the model answers with the option letter, not with "True".**
Without chat template: `' ('` 0.70, `' A'` 0.29, `' True'` only 0.001; `'True'` (id 2514, the token
LM-Polygraph scores) is not even in the top 10. With chat template: `'The'` 0.48, `'(A'` 0.37.
So reading the score of "True" gives almost no signal for this model. Instead, we end the prompt
with `" ("` and compare the tokens `A` and `B`.

### Step 3: A/B letter version of p(True)

> **Note (Day 2):** Cells in this step are the original exploration code (`p_true_letter`). The final implementation is the shared `common.reflexive` module (`mode="plain_ab"`), which is used in the sanity test cells below. Both give identical numbers on Qwen 0.5B and 1.5B.

In [4]:
def brainstorm_prompt(question, answer, samples):
    return (
        f"Question: {question}\n"
        f"Here are some ideas that were brainstormed: {', '.join(samples)}\n"
        f"Possible answer: {answer}\n"
        "Is the possible answer:\n"
        " (A) True\n"
        " (B) False\n"
        "The possible answer is: ("
    )

id_A = tok("A", add_special_tokens=False)["input_ids"]
id_B = tok("B", add_special_tokens=False)["input_ids"]
print("A ->", id_A, " B ->", id_B)

def p_true_letter(question, answer, samples):
    ids = tok(brainstorm_prompt(question, answer, samples), return_tensors="pt")
    with torch.no_grad():
        logits = model(**ids).logits[0, -1]
    pair = torch.stack([logits[id_A[0]], logits[id_B[0]]])
    return torch.softmax(pair, dim=0)[0].item()   # P(A) / (P(A) + P(B))

samples = ["Paris", "Paris", "Lyon", "Paris", "Marseille"]
for ans in ["Paris", "Lyon", "Berlin"]:
    print(f"{ans:>8}: p(True) = {p_true_letter('What is the capital of France?', ans, samples):.3f}")

A -> [32]  B -> [33]
   Paris: p(True) = 1.000
    Lyon: p(True) = 0.957
  Berlin: p(True) = 0.164


In [11]:
def ab_coverage(question, answer, samples=None):
    ids = tok(brainstorm_prompt(question, answer, samples), return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**ids).logits[0, -1].float(), dim=0)
    return probs[id_A[0]].item() + probs[id_B[0]].item()

for ans in ["Paris", "Berlin"]:
    print(f"{ans:>7}: coverage = {ab_coverage('What is the capital of France?', ans):.3f}")

  Paris: coverage = 1.000
 Berlin: coverage = 1.000


**Coverage of the A/B method.** Coverage = P(A) + P(B) for the next token, i.e. how much of the model's
probability our two tokens capture. Qwen 0.5B, France question: Paris 1.000, Berlin 1.000.
Compare R1 (Qwen 0.5B): plain prompt 0.02–0.08, chat template + token groups 0.67–0.75.
Ending the prompt with " (" makes the model put essentially all its probability on A or B,
so we read its full decision. Coverage says we read the right token, not that the decision is good.


### Step 4: How do brainstormed samples affect p(True)?

In [5]:
def brainstorm_prompt(question, answer, samples=None):
    brainstorm = (f"Here are some ideas that were brainstormed: {', '.join(samples)}\n"
                  if samples else "")
    return (
        f"Question: {question}\n"
        f"{brainstorm}"
        f"Possible answer: {answer}\n"
        "Is the possible answer:\n"
        " (A) True\n"
        " (B) False\n"
        "The possible answer is: ("
    )

q = "What is the capital of France?"
conditions = {
    "no brainstorm":              None,
    "samples with Lyon":          ["Paris", "Paris", "Lyon", "Paris", "Marseille"],
    "samples all Paris":          ["Paris", "Paris", "Paris", "Paris", "Paris"],
    "samples all Lyon":           ["Lyon", "Lyon", "Lyon", "Lyon", "Lyon"],
}
for name, s in conditions.items():
    print(name)
    for ans in ["Paris", "Lyon", "Berlin"]:
        print(f"   {ans:>8}: {p_true_letter(q, ans, s):.3f}")

no brainstorm
      Paris: 0.961
       Lyon: 0.562
     Berlin: 0.245
samples with Lyon
      Paris: 1.000
       Lyon: 0.957
     Berlin: 0.164
samples all Paris
      Paris: 0.996
       Lyon: 0.562
     Berlin: 0.119
samples all Lyon
      Paris: 0.926
       Lyon: 0.996
     Berlin: 0.202


**Experiment: how do brainstormed samples affect p(True)?** (Qwen2.5-0.5B, one question, hand-written samples)

| Condition | Paris (correct) | Lyon (wrong) | Berlin (wrong) |
|---|---|---|---|
| No brainstorm | 0.961 | 0.562 | 0.245 |
| Samples contain Lyon | 1.000 | 0.957 | 0.164 |
| Samples all Paris | 0.996 | 0.562 | 0.119 |
| Samples all Lyon | 0.926 | 0.996 | 0.202 |

- Without brainstorm, the model's own knowledge is weak but points the right way (Lyon 0.56 vs Paris 0.96).
- An answer that appears in the samples is strongly boosted (Lyon 0.56 → 0.96 → 0.996).
- The effect is one-directional: answers that are absent from the samples are not lowered (Lyon stays at 0.56 when all samples say Paris).
- Hypothesis: with real model samples, brainstorm p(True) may confirm the model's own frequent mistakes and behave like a consistency measure (similar to semantic entropy). To test: correlation with semantic entropy on the 50-question data.
- Caveat: one question, hand-written samples, small model.

### Step 5: Shared sanity test (`common/sanity.py`), checked with dummy functions

In [6]:
%load_ext autoreload
%autoreload 2
from common.sanity import SANITY, run_sanity, save_sanity

correct_set = {(q, r) for q, r, _ in SANITY}

perfect  = lambda q, a, s: 1.0 if (q, a) in correct_set else 0.0
inverse  = lambda q, a, s: 0.0 if (q, a) in correct_set else 1.0
constant = lambda q, a, s: 0.9

for name, fn in [("perfect", perfect), ("inverse", inverse), ("constant", constant)]:
    r = run_sanity(fn)
    print(f"{name:>8}: auroc={r['auroc']:.2f}  gap={r['gap']:.2f}  n_distinct={r['n_distinct']}")

 perfect: auroc=1.00  gap=1.00  n_distinct=2
 inverse: auroc=0.00  gap=-1.00  n_distinct=2
constant: auroc=0.50  gap=0.00  n_distinct=1


## Results

> **Update (Day 2, shared code):** The p(True) code in this notebook (`p_true_letter`) is the original R2 implementation and is kept as a record of how the method was developed. It is now part of the shared module `common/reflexive.py` as `mode="plain_ab"`, which both zero-shot (R1) and brainstorm (R2) p(True) use; the only difference between them is the brainstorm line. The shared code reproduces the results below exactly on Qwen 0.5B and 1.5B. The mode comparison and decision (plain_ab chosen) are in `notebooks/reflexive_mode_comparison.ipynb`. All new runs (Day 3 onwards) use `common.reflexive.p_true(..., mode="plain_ab")`.

### Sanity test on Qwen2.5-0.5B: without and with brainstorm

In [7]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

r_none = run_sanity(lambda q, a, s: p_true_letter(q, a, None))
print(f"no brainstorm: auroc={r_none['auroc']:.2f}  gap={r_none['gap']:.2f}  n_distinct={r_none['n_distinct']}")
save_sanity(r_none, "r2_letter_nobrainstorm", MODEL)

no brainstorm: auroc=0.76  gap=0.15  n_distinct=15


PosixPath('results/sanity/r2_letter_nobrainstorm_Qwen2.5-0.5B-Instruct.json')

In [8]:
SAMPLES_PATH = "data/qwen0.5b/sanity_samples.json"

if os.path.exists(SAMPLES_PATH):
    with open(SAMPLES_PATH) as f:
        samples = json.load(f)
    print("Loaded saved samples from", SAMPLES_PATH)
else:
    sampler = HFTextGenerationSampler(model, tok, num_samples=10, temperature=0.7, max_new_tokens=16)
    prompts = ["Answer the following question with as few words as possible. " + q for q, _, _ in SANITY]
    texts = sampler(prompts).tensor.text
    samples = {q: [str(t).strip().split("\n")[0] for t in texts[i]] for i, (q, _, _) in enumerate(SANITY)}
    with open(SAMPLES_PATH, "w") as f:
        json.dump(samples, f, indent=2)
    print("Generated and saved samples to", SAMPLES_PATH)

for q in list(samples)[:2]:
    print(q, "->", samples[q])

Loaded saved samples from data/qwen0.5b/sanity_samples.json
What is the capital of France? -> ['Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris']
What is the chemical symbol for water? -> ['H2O', 'H2O', 'H2O', 'H2O', 'H2O', 'H2O', 'H2O', 'H2O', 'H2O', 'H2O']


In [9]:
r_brain = run_sanity(p_true_letter, samples=samples)
print(f"brainstorm:    auroc={r_brain['auroc']:.2f}  gap={r_brain['gap']:.2f}  n_distinct={r_brain['n_distinct']}")
save_sanity(r_brain, "r2_letter_brainstorm", MODEL)

brainstorm:    auroc=1.00  gap=0.51  n_distinct=16


PosixPath('results/sanity/r2_letter_brainstorm_Qwen2.5-0.5B-Instruct.json')

In [10]:
for q, s in samples.items():
    print(f"{q[:45]:<45} -> {sorted(set(s))}")

What is the capital of France?                -> ['Paris']
What is the chemical symbol for water?        -> ['H2O']
How many planets are in our solar system?     -> ['Our solar system has 8 planets: Earth, Mars, Jupiter, Saturn,', 'Our solar system has 8 planets: Mercury, Venus, Earth, Mars,', 'Our solar system has eight planets: Mercury, Venus, Earth, Mars, Jupiter']
Who wrote Romeo and Juliet?                   -> ['Shakespeare wrote Romeo and Juliet.', 'Shakespeare.', 'William Shakespeare', 'William Shakespeare wrote "Romeo and Juliet".', 'William Shakespeare wrote Romeo and Juliet.']
What is the largest ocean on Earth?           -> ['The largest ocean on Earth is the Pacific Ocean.']
At what temperature in Celsius does water boi -> ['100 Celsius', '100 Celsius.', '100°C', 'At 100°C (212°F) in Celsius, water', 'At 100°C (212°F), water boils at sea']
Who was the first person to walk on the Moon? -> ['Gordian Spacewalk was the first lunar mission.', 'Neil Armstrong was the first perso

**Note on the sampled answers.** Even with "as few words as possible", Qwen 0.5B sometimes answers
with full sentences ("The largest ocean on Earth is the Pacific Ocean."), and some samples are cut off
by `max_new_tokens=16` ("At 100°C (21"). All samples are still correct, which again shows that the
sanity test favors brainstorm. Long samples make the brainstorm prompt longer; to keep in mind for the
50-question data (32 new tokens).

**Sanity test, Qwen2.5-0.5B, A/B letter version of p(True)** (prompt ends with " (", softmax over "A" vs "B", no chat template)

| Method | AUROC | Gap | n_distinct |
|---|---|---|---|
| No brainstorm | 0.76 | 0.15 | 15 |
| Brainstorm (10 model samples, temperature 0.7) | 1.00 | 0.51 | 16 |

- Without brainstorm the signal is weak but real.
- With brainstorm the separation is perfect. On these easy questions the model's samples are almost always the correct answer
  (e.g. 10× "Paris", 10× "H2O"), while our wrong answers never appear in the samples.
- So in the sanity test, brainstorm effectively checks "is this answer among the model's own samples?". The test favors brainstorm by design.
- On real data the evaluated answer is the model's own greedy answer; if the model consistently produces a wrong answer,
  brainstorm may confirm it. To test on the 50 TriviaQA questions, together with the correlation with semantic entropy.

### Sanity test on all three models

In [12]:
import gc
from pathlib import Path

MODELS = {
    "smollm135m": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "qwen0.5b":   "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen1.5b":   "Qwen/Qwen2.5-1.5B-Instruct",
}

def run_model(key, name):
    m, t = load_model(name)
    m.eval()
    a = t("A", add_special_tokens=False)["input_ids"]
    b = t("B", add_special_tokens=False)["input_ids"]
    assert len(a) == 1 and len(b) == 1, f"A or B is not a single token: {a}, {b}"

    def next_probs(text):
        ids = t(text, return_tensors="pt")
        with torch.no_grad():
            return torch.softmax(m(**ids).logits[0, -1].float(), dim=0)

    def p_fn(q, ans, s):
        p = next_probs(brainstorm_prompt(q, ans, s))
        return (p[a[0]] / (p[a[0]] + p[b[0]])).item()

    # Coverage on all 20 sanity items, without brainstorm
    cov = []
    for q, right, wrong in SANITY:
        for ans in (right, wrong):
            p = next_probs(brainstorm_prompt(q, ans, None))
            cov.append((p[a[0]] + p[b[0]]).item())

    # Sanity samples: load if saved, otherwise generate once and save
    path = Path(f"data/{key}/sanity_samples.json")
    if path.exists():
        samples_m = json.loads(path.read_text())
    else:
        sampler = HFTextGenerationSampler(m, t, num_samples=10, temperature=0.7, max_new_tokens=16)
        prompts = ["Answer the following question with as few words as possible. " + q for q, _, _ in SANITY]
        texts = sampler(prompts).tensor.text
        samples_m = {q: [str(x).strip().split("\n")[0] for x in texts[i]] for i, (q, _, _) in enumerate(SANITY)}
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(samples_m, indent=2))

    r_none = run_sanity(lambda q, ans, s: p_fn(q, ans, None))
    r_brain = run_sanity(p_fn, samples=samples_m)
    save_sanity(r_none, "r2_letter_nobrainstorm", name)
    save_sanity(r_brain, "r2_letter_brainstorm", name)

    del m, t
    gc.collect()
    return {"model": key, "coverage_min": min(cov), "coverage_mean": float(np.mean(cov)),
            "auroc_none": r_none["auroc"], "gap_none": r_none["gap"],
            "auroc_brain": r_brain["auroc"], "gap_brain": r_brain["gap"]}

table = []
for key, name in MODELS.items():
    print("running", key, "...")
    table.append(run_model(key, name))

print(f"\n{'model':<12} {'cov min':>8} {'cov mean':>9} {'AUROC none':>11} {'gap none':>9} {'AUROC brain':>12} {'gap brain':>10}")
for r in table:
    print(f"{r['model']:<12} {r['coverage_min']:>8.3f} {r['coverage_mean']:>9.3f} {r['auroc_none']:>11.2f} "
          f"{r['gap_none']:>9.2f} {r['auroc_brain']:>12.2f} {r['gap_brain']:>10.2f}")
    

running smollm135m ...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

running qwen0.5b ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

running qwen1.5b ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


model         cov min  cov mean  AUROC none  gap none  AUROC brain  gap brain
smollm135m      0.637     0.720        0.37     -0.01         0.34      -0.04
qwen0.5b        1.000     1.000        0.76      0.15         1.00       0.52
qwen1.5b        0.999     1.000        0.99      0.78         1.00       0.83


**Sanity test on three models (A/B letter method, plain prompt ending in " (")**

| Model | Coverage min / mean | AUROC no brainstorm | Gap | AUROC brainstorm | Gap |
|---|---|---|---|---|---|
| SmolLM2-135M | 0.64 / 0.72 | 0.37 | -0.01 | 0.34 | -0.04 |
| Qwen2.5-0.5B | 1.00 / 1.00 | 0.76 | 0.15 | 1.00 | 0.52 |
| Qwen2.5-1.5B | 1.00 / 1.00 | 0.99 | 0.78 | 1.00 | 0.83 |

- **SmolLM2-135M:** the method does not work (AUROC below 0.5, negative gap), and brainstorm does not help. Consistent with R1's finding that the letter route points the wrong way for SmolLM. The model is too small to judge its own answers.
- **Qwen 0.5B:** weak without brainstorm, perfect with it. Brainstorm helps most here. (Same as before; the gap 0.51 → 0.52 comes from computing the probabilities in float32.)
- **Qwen 1.5B:** nearly perfect even without brainstorm; brainstorm adds little.
- p(True) improves with model size, as reported by Kuhn et al. and Farquhar et al.
- Feasibility: reflexive methods work on Qwen 0.5B (with brainstorm) and Qwen 1.5B.
- Caveat: 20 easy items; the sanity test favors brainstorm. R1's 50-question result on Qwen 0.5B (zero-shot) is much weaker (mean p(True) 0.90 correct vs 0.855 wrong).

## Day 3: Brainstorm p(True) on 50 TriviaQA questions (Qwen 0.5B)

Shared code: `common.reflexive.p_true_details(..., mode="plain_ab")`. Each question's greedy answer is scored twice with the same prompt, without and with the brainstorm line (the 10 samples from `load_data`). Only the brainstorm version is saved as the official score file (`p_true_brainstorm.json`); zero-shot is kept in the details file for the paired comparison.


In [2]:
import json, time, math
from pathlib import Path
import numpy as np
from common.data import load_data
from common.reflexive import p_true_details
from common.io import save_scores

data = load_data("qwen0.5b", 50)
print(len(data), "questions, accuracy:", np.mean([r["correct"] for r in data]))
print("samples per question:", sorted({len(r["samples"]) for r in data}))
print("samples containing a newline:", sum("\n" in s for r in data for s in r["samples"]))

50 questions, accuracy: 0.24
samples per question: [10]
samples containing a newline: 0


In [3]:
from probly.representer.sampler.huggingface import load_model

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
model, tok = load_model(MODEL)
model.eval()

rows = []
t_brain = 0.0
for r in data:
    z = p_true_details(model, tok, r["question"], r["greedy_answer"], mode="plain_ab")
    t0 = time.perf_counter()
    b = p_true_details(model, tok, r["question"], r["greedy_answer"], brainstorm=r["samples"], mode="plain_ab")
    t_brain += time.perf_counter() - t0
    rows.append({"id": r["id"], "correct": r["correct"],
                 "p_true_zeroshot": z["p_true"], "coverage_zeroshot": z["coverage"],
                 "p_true_brainstorm": b["p_true"], "coverage_brainstorm": b["coverage"]})

print(f"done, {t_brain / len(rows):.2f} s per question (brainstorm)")
print("mean coverage zero-shot / brainstorm:",
      round(np.mean([x["coverage_zeroshot"] for x in rows]), 3),
      round(np.mean([x["coverage_brainstorm"] for x in rows]), 3))
for name in ["p_true_zeroshot", "p_true_brainstorm"]:
    pc = np.mean([x[name] for x in rows if x["correct"]])
    pw = np.mean([x[name] for x in rows if not x["correct"]])
    print(f"{name:<18} mean p(True): correct={pc:.3f}  wrong={pw:.3f}  gap={pc - pw:+.3f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

done, 2.37 s per question (brainstorm)
mean coverage zero-shot / brainstorm: 1.0 1.0
p_true_zeroshot    mean p(True): correct=0.966  wrong=0.882  gap=+0.083
p_true_brainstorm  mean p(True): correct=0.985  wrong=0.826  gap=+0.159


In [4]:
unc, failed = [], []
for x in rows:
    p = x["p_true_brainstorm"]
    ok = p is not None and math.isfinite(p)
    unc.append(1 - p if ok else 1.0)
    failed.append(not ok)

record = {"method": "p_true_brainstorm", "model": MODEL, "answer": "greedy",
          "ids": [x["id"] for x in rows], "uncertainty": unc, "failed": failed,
          "seconds_per_question": t_brain / len(rows)}
path = save_scores(record, run="qwen0.5b", expected_ids=[r["id"] for r in data])
print("saved:", path)

details_path = Path("results/qwen0.5b/p_true_brainstorm_details.json")
details_path.write_text(json.dumps(rows, indent=2))
print("saved:", details_path)

saved: /Users/elcinekinsimsir/projects/probly-fork/experiments/llm-probly/results/qwen0.5b/p_true_brainstorm.json
saved: results/qwen0.5b/p_true_brainstorm_details.json


In [5]:
from sklearn.metrics import roc_auc_score
from common.evaluate import evaluate, bootstrap_auroc
from common.io import load_scores

ids = [r["id"] for r in data]
correct = np.array([r["correct"] for r in data], dtype=bool)
assert [x["id"] for x in rows] == ids

scores = {
    "p_true_zeroshot (plain_ab)": np.array([1 - x["p_true_zeroshot"] for x in rows]),
    "p_true_brainstorm (plain_ab)": np.array([1 - x["p_true_brainstorm"] for x in rows]),
}
for f in sorted(Path("results/qwen0.5b").glob("*.json")):
    if f.name.endswith("_details.json") or f.stem == "p_true_brainstorm":
        continue
    try:
        rec = load_scores(f, expected_ids=ids)
        scores[rec["method"]] = np.array(rec["uncertainty"], dtype=float)
    except Exception as e:
        print("skipped", f.name, "->", e)

print(f"\n{'method':<30} {'AUROC':>6} {'95% CI':>16} {'AURC':>6}")
for name, u in scores.items():
    m = evaluate(u, correct)
    lo, hi = bootstrap_auroc(u, correct)
    print(f"{name:<30} {m['auroc']:>6.3f} [{lo:.3f}, {hi:.3f}] {m['aurc']:>6.3f}")

wrong = ~correct
uz = scores["p_true_zeroshot (plain_ab)"]
ub = scores["p_true_brainstorm (plain_ab)"]
rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    idx = rng.integers(0, len(wrong), len(wrong))
    if wrong[idx].all() or not wrong[idx].any():
        continue
    diffs.append(roc_auc_score(wrong[idx], ub[idx]) - roc_auc_score(wrong[idx], uz[idx]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nPaired bootstrap, AUROC brainstorm - zero-shot: {np.mean(diffs):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")


method                          AUROC           95% CI   AURC
p_true_zeroshot (plain_ab)      0.678 [0.480, 0.862]  0.666
p_true_brainstorm (plain_ab)    0.812 [0.679, 0.933]  0.603
likelihood_mean                 0.814 [0.656, 0.943]  0.605
likelihood_sum                  0.662 [0.450, 0.848]  0.664
p_true_empirical                0.531 [0.355, 0.710]  0.746
p_true_zeroshot_norm            0.654 [0.449, 0.834]  0.672
verbalized_2s                   0.432 [0.268, 0.608]  0.793

Paired bootstrap, AUROC brainstorm - zero-shot: +0.133  95% CI [-0.041, +0.330]


In [6]:
from scipy.stats import spearmanr
from common.correctness import normalize

agree = np.array([np.mean([normalize(s) == normalize(r["greedy_answer"]) for s in r["samples"]]) for r in data])
pz = np.array([x["p_true_zeroshot"] for x in rows])
pb = np.array([x["p_true_brainstorm"] for x in rows])

print("Spearman correlation with sample agreement:")
print(f"  zero-shot p(True):  {spearmanr(pz, agree).statistic:+.3f}")
print(f"  brainstorm p(True): {spearmanr(pb, agree).statistic:+.3f}")
print(f"  correct (0/1):      {spearmanr(correct, agree).statistic:+.3f}")

print("\nConsistently wrong (wrong answer, >= 70% of samples agree):")
for r, a, z, b in zip(data, agree, pz, pb):
    if not r["correct"] and a >= 0.7:
        print(f"  id={r['id']:>2}  agree={a:.1f}  p_zeroshot={z:.3f}  p_brainstorm={b:.3f}  answer={r['greedy_answer']!r}")

Spearman correlation with sample agreement:
  zero-shot p(True):  +0.148
  brainstorm p(True): +0.573
  correct (0/1):      +0.277

Consistently wrong (wrong answer, >= 70% of samples agree):
  id= 0  agree=0.7  p_zeroshot=0.984  p_brainstorm=0.998  answer='United Kingdom'
  id= 9  agree=0.7  p_zeroshot=0.999  p_brainstorm=0.971  answer='The first commercial jet airliner to go into service in the UK was the Boeing 707.'
  id=10  agree=0.7  p_zeroshot=0.984  p_brainstorm=0.986  answer='Vietnam'
  id=44  agree=0.8  p_zeroshot=0.469  p_brainstorm=0.998  answer='Vincent van Gogh'


In [7]:
u_agree = 1 - agree
m = evaluate(u_agree, correct)
lo, hi = bootstrap_auroc(u_agree, correct)
print(f"{'sample agreement (1 - agree)':<30} AUROC={m['auroc']:.3f} [{lo:.3f}, {hi:.3f}]  AURC={m['aurc']:.3f}")
print("distinct agreement values:", len(np.unique(agree)))

rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    idx = rng.integers(0, len(wrong), len(wrong))
    if wrong[idx].all() or not wrong[idx].any():
        continue
    diffs.append(roc_auc_score(wrong[idx], ub[idx]) - roc_auc_score(wrong[idx], u_agree[idx]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Paired bootstrap, AUROC brainstorm p(True) - agreement: {np.mean(diffs):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

sample agreement (1 - agree)   AUROC=0.684 [0.483, 0.867]  AURC=0.643
distinct agreement values: 10
Paired bootstrap, AUROC brainstorm p(True) - agreement: +0.130  95% CI [-0.041, +0.320]


### Day 3 results (Qwen 0.5B, 50 TriviaQA questions, 12 correct)

| Uncertainty score | AUROC [95% CI] | AURC |
|---|---|---|
| p(True) zero-shot (plain_ab) | 0.678 [0.480, 0.862] | 0.666 |
| Sample agreement (1 − share of samples equal to greedy) | 0.684 [0.483, 0.867] | 0.643 |
| **p(True) brainstorm (plain_ab)** | **0.812 [0.679, 0.933]** | **0.603** |
| likelihood_mean (baseline) | 0.814 [0.656, 0.943] | 0.605 |

**Findings**
- Brainstorm improves p(True): +0.133 AUROC over zero-shot (paired bootstrap 95% CI [−0.041, +0.330]). The direction is clear, but with 50 questions the difference is not significant.
- Brainstorm p(True) is strongly driven by consistency: its Spearman correlation with sample agreement is +0.573 (zero-shot: +0.148). This confirms the Lyon experiment on real data.
- It is not *only* consistency: sample agreement alone reaches 0.684, about the same as zero-shot; brainstorm p(True) is +0.130 better than agreement (CI [−0.041, +0.320]). It seems to combine the model's own judgment with the consistency of the samples.
- Failure mode: consistently wrong answers get confirmed. Example id 44 ("Vincent van Gogh", wrong, 8/10 samples agree): zero-shot p(True) 0.469 (rightly doubtful) → brainstorm 0.998.
- Brainstorm p(True) only matches likelihood_mean, which is almost free, while brainstorm needs 10 sampled answers plus an extra forward pass (2.37 s/question for p(True) alone, sampling time not included).

**Caveats**
- Only 12 correct answers, so all confidence intervals are wide.
- Sample agreement uses exact matching after normalization ("UK" ≠ "United Kingdom"). Semantic entropy clusters by meaning and may be a stronger consistency signal; the main comparison with SE is still to do once those scores are available.
- Coverage is 1.00 in both conditions, so plain_ab works on real data as on the sanity test.

## What works / What does not

**Works**
- Scoring the option letters (`A` vs `B`) after a prompt ending in `" ("`: both are single tokens (32, 33) and give a clear signal.
- `run_sanity()` gives the expected values on dummy functions (perfect 1.00, inverse 0.00, constant 0.50).
- Brainstorm p(True) separates correct and wrong answers perfectly in the sanity test (AUROC 1.00 vs 0.76 without brainstorm).
- Sanity samples are saved and reloaded, so the results are reproducible.

**Does not work / limitations**
- LM-Polygraph's approach (raw score of the token "True" without a space) reads a token Qwen 0.5B practically never produces.
- The chat template changes the first predicted token ("The..."), so single-token scoring is harder with it.
- The sanity test favors brainstorm: our wrong answers never appear in the model's own samples.
- The brainstorm experiment (Step 4) uses one question and hand-written samples: an indication, not a result.
- Samples are sometimes full sentences or cut off by the token limit.

## Questions for tutors

1. Should p(True) score the option letter (A/B) instead of the "True" token? With or without chat template?
2. Brainstorm p(True) seems to reward answers that appear in the model's own samples. Is it fair to compare it with semantic entropy, or is it partly a consistency measure itself?
3. Where should p(True) live in probly: as a representer (it queries the model again) or as a measure? Since entropy is symmetric, would a measure like `1 - p(True)` be needed?
4. Brainstorm p(True) correlates strongly with sample consistency but adds information beyond exact-match agreement. Is comparing against semantic entropy the right way to separate "judgment" from "consistency", or would you suggest another control (e.g. shuffled or random samples in the brainstorm line)?